# Cortical Connectome Eigenmodes

---

This notebook will use the spectranorm python package to map the connectome eigenmodes that are used as an orthonormal spectral basis set for spectral 
normative modeling.


### package imports and basic functions

---

In [1]:
import os
import gc
import sys
import glob
import shutil
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
import zipfile
from pathlib import Path
import polars as pl
import colorcet as cc


In [2]:
%load_ext autoreload
%autoreload 2

# Path to add to src folder (to use local version of spectranorm)
sys.path.append(os.path.abspath("/mountpoint/code/projects/spectranorm/package/spectranorm/src/"))

from spectranorm import snm


In [3]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


## Load fs-LR 32k cortical templates

---

In [4]:
# basic parameters
surface = 'midthickness_MSMAll'

# load an example dscalar
hcp_pipeline_dir = "/mountpoint/data/HCPpipelines"
dscalar_file = f"{hcp_pipeline_dir}/global/templates/91282_Greyordinates/91282_Greyordinates.dscalar.nii"
dscalar = nib.load(dscalar_file)

brain_models = [x for x in dscalar.header.get_index_map(1).brain_models]

# create a mapping between surface and cifti vertices
left_cortical_surface_model, right_cortical_surface_model = brain_models[0], brain_models[1]
cifti_to_surface = {}
surface_to_cifti = {}
for (i, x) in enumerate(left_cortical_surface_model.vertex_indices):
    cifti_to_surface[i] = x
    surface_to_cifti[x] = i
for (i, x) in enumerate(right_cortical_surface_model.vertex_indices):
    cifti_to_surface[i + right_cortical_surface_model.index_offset] = x + left_cortical_surface_model.surface_number_of_vertices
    surface_to_cifti[x + left_cortical_surface_model.surface_number_of_vertices] = i + right_cortical_surface_model.index_offset

# construct data over surface
surface_mask = list(surface_to_cifti.keys())

left_surface_mask = surface_mask[:right_cortical_surface_model.index_offset]
right_surface_mask = surface_mask[right_cortical_surface_model.index_offset:]


## Load high-resolution structural connectomes

---

In [5]:
%%time
# number of vertices in cifti cortical space
cifti_n_verts = np.array(left_cortical_surface_model.vertex_indices).shape[0] + np.array(right_cortical_surface_model.vertex_indices).shape[0]

# load the smoothed connectome data in cifti space
smoothed_connectome_cifti = sparse.load_npz(
    "/mountpoint/data/connectomes/HCP_1200-high_resolution_cifti_32k_MSMAll-thresholded_smoothed_group_connectome_streamline_count.npz"
)[:cifti_n_verts, :cifti_n_verts]
smoothed_connectome_cifti


CPU times: user 29.5 s, sys: 22.1 s, total: 51.7 s
Wall time: 4min 12s


<Compressed Sparse Column sparse matrix of dtype 'float64'
	with 464169158 stored elements and shape (59412, 59412)>

## Compute random walk Laplacian eigenmodes

---

In [6]:
%%time
# compute eigenmodes of the smoothed connectome at this resolution
eigenmodes_cifti_cortex = snm.utils.gsp.compute_random_walk_laplacian_eigenmodes(
    adjacency_matrix=smoothed_connectome_cifti,
    num_eigenvalues=11000,
)


CPU times: user 12d 16h 57min 50s, sys: 3min 31s, total: 12d 17h 1min 22s
Wall time: 12h 51min 18s


In [24]:
%%time
# store the omputed eigenmodes as an instance of EigenmodeBasis
eigenmode_basis = snm.utils.gsp.EigenmodeBasis(
    eigenvalues=eigenmodes_cifti_cortex[0],
    eigenvectors=eigenmodes_cifti_cortex[1].T,
    mass_matrix=sparse.diags(eigenmodes_cifti_cortex[2])
)


CPU times: user 717 μs, sys: 0 ns, total: 717 μs
Wall time: 638 μs


In [26]:
%%time
# save the basis for future use
eigenmode_basis.save(
    "/mountpoint/data/connectomes/HCP_1200-high_resolution_cifti_32k_MSMAll-connectome-Lrw-eigenmodes.joblib"
)


CPU times: user 341 ms, sys: 3.88 s, total: 4.22 s
Wall time: 4.24 s


## Scratch

---

In [ ]:
eigenvalues = np.load("/mountpoint/code/projects/normative_brain_charts/data/npy/rw_cortical_connectome_eigenvalues_cifti.npy")
eigenvectors = np.load("/mountpoint/code/projects/normative_brain_charts/data/npy/rw_cortical_connectome_eigenvectors_cifti.npy")

# store the computed eigenmodes as an instance of EigenmodeBasis
eigenmode_basis = snm.utils.gsp.EigenmodeBasis(
    eigenvalues=eigenvalues[:10000],
    eigenvectors=eigenvectors[:, :10000],
)

# save the basis for future use
eigenmode_basis.save(
    "/mountpoint/code/projects/normative_brain_charts/data/rw_cortical_connectome_eigenmode_basis_cifti.joblib"
)